In [ ]:
import pandas as pd
import os
from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split

INDIA_PATH = Path(r"C:\Users\Asus\Desktop\Anemia-Detection-MedGammaGoogle\Anemia-Detection-MedGammaGoogle\data\raw\dataset anemia\India")
PROCESSED_PATH = Path(r"C:\Users\Asus\Desktop\Anemia-Detection-MedGammaGoogle\Anemia-Detection-MedGammaGoogle\data\processed")

In [ ]:
# Load India Excel, clean Hgb, apply severity
india_df = pd.read_excel(INDIA_PATH / "India.xlsx")
india_df['Hgb'] = pd.to_numeric(india_df['Hgb'], errors='coerce')
india_df = india_df.dropna(subset=['Hgb'])

def map_severity(hgb):
    if hgb < 8:
        return "Severe"
    elif hgb < 11:
        return "Moderate"
    elif hgb < 12:
        return "Mild"
    else:
        return "None"

india_df['severity'] = india_df['Hgb'].apply(map_severity)
print(india_df.shape)
print(india_df['severity'].value_counts())

In [ ]:
# Find JPG for each patient
def findjpg(patient_number):
    folder_path = INDIA_PATH / str(patient_number)
    for f in os.listdir(folder_path):
        if f.endswith('.jpg'):
            return str(os.path.join(folder_path, f))
    return None

india_df['image_path'] = india_df['Number'].apply(findjpg)
print(india_df[['Number', 'image_path']].head(10))
print("Missing images:", india_df['image_path'].isna().sum())

In [ ]:
# Resize images to 224x224 and save to data/processed/images/
# Assign directly by index to avoid misalignment after concat
india_df['processed_path'] = None

for idx, row in india_df.iterrows():
    patient_image = Image.open(row['image_path'])
    resized_image = patient_image.resize((224, 224), Image.LANCZOS)
    new_path = PROCESSED_PATH / "images" / f"{row['Number']}.jpg"
    resized_image.save(new_path)
    india_df.at[idx, 'processed_path'] = str(new_path)

print("Total processed:", india_df['processed_path'].notna().sum())
print(india_df[['Number', 'processed_path']].head())

In [ ]:
# 70/15/15 train/val/test split stratified on severity
train_df, temp_df = train_test_split(
    india_df,
    test_size=0.3,
    stratify=india_df['severity'],
    random_state=42
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=None,
    random_state=42
)

train_df = train_df.copy()
valid_df = valid_df.copy()
test_df = test_df.copy()

train_df['split'] = 'train'
valid_df['split'] = 'valid'
test_df['split'] = 'test'

india_df = pd.concat([train_df, valid_df, test_df])
print(india_df['split'].value_counts())

In [ ]:
# Save master.csv
india_df_selected = india_df[['processed_path', 'Number', 'Hgb', 'severity', 'split']]
india_df_selected.to_csv(PROCESSED_PATH / "master.csv", index=False)

# Read back with keep_default_na=False so 'None' string is not read as NaN
test = pd.read_csv(PROCESSED_PATH / "master.csv", keep_default_na=False)
print(test.shape)
print(test.head(10))
print("\nSplit distribution:")
print(test['split'].value_counts())
print("\nSeverity distribution:")
print(test['severity'].value_counts())
print("\nNaN severity rows:", test[test['severity'] == ''].shape[0])